In [1]:
!pip install black==24.3.0 --no-deps
!pip install pylint==2.17.7 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 127.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.2/537.2 kB 53.7 MB/s  0:00:00


# Configuracion

In [ ]:
team = 'RETAIL'                  # Equipo: RETAIL
name_ds = 'Hernandez Santiago'   # Apellidos y nombres del responsable
path_script = './inference.py'


In [3]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


# Formatear Script

In [4]:
!black --line-length 100 $path_script
!pylint --disable C0103 $path_script

Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/black", line 3, in <module>
    from black import patched_main
  File "src/black/__init__.py", line 33, in <module>
ModuleNotFoundError: No module named 'mypy_extensions'
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/pylint", line 6, in <module>
    sys.exit(run_pylint())
             ^^^^^^^^^^^^
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/__init__.py", line 33, in run_pylint
    from pylint.lint import Run as PylintRun
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/lint/__init__.py", line 19, in <module>
    from pylint.config.exceptions import ArgumentPreprocessingError
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/config/__init__.py", line 25, in <module>
    from pylint.config.arguments_provider import UnsupportedAction
  File "/home/ec2-user/anaconda3

# Crear Processor

| Instancia      | CPU Cores | Memoria (GB) |
|----------------|-----------|--------------|
| ml.m5.large    | 2         | 8            |
| ml.m5.xlarge   | 4         | 16           |
| ml.m5.2xlarge  | 8         | 32           |
| ml.m5.4xlarge  | 16        | 64           |
| ml.m5.12xlarge | 48        | 192          |
| ml.m5.24xlarge | 96        | 384          |

In [5]:
from sagemaker.processing import ScriptProcessor

processor = ScriptProcessor(instance_type='ml.m5.12xlarge',
                            volume_size_in_gb=30,
                            instance_count=1,
                            command=['python3'],
                            image_uri=image_uri,
                            role=role_arn,
                            tags=tags)

# Definir Entradas

In [ ]:
from sagemaker.processing import ProcessingInput

S3_BASE = 's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'
PERIODO = '202604'  # ← actualizar al periodo de inferencia

inputs = []

inputs.append(ProcessingInput(
    input_name='code/utils',
    source=f'{S3_BASE}/utils/',
    destination='/opt/ml/processing/input/code/utils',
))

inputs.append(ProcessingInput(
    input_name='data',
    source=f'{S3_BASE}/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo={PERIODO}/',
    destination='/opt/ml/processing/input/data',
))

inputs.append(ProcessingInput(
    input_name='models',
    source=f'{S3_BASE}/MODEL/calibrado_v2/',
    destination='/opt/ml/processing/input/models',
))

inputs.append(ProcessingInput(
    input_name='artifacts',
    source=f'{S3_BASE}/MODEL/artifacts_v2/',  # encoding_maps.json + percentil_limits.json
    destination='/opt/ml/processing/input/artifacts',
))


# Definir Salidas

In [ ]:
from sagemaker.processing import ProcessingOutput

outputs = []

outputs.append(ProcessingOutput(
    output_name='results',
    source='/opt/ml/processing/output/results',
    destination=f'{S3_BASE}/REPLICA_OUTPUT',
))


# Ejecutar Job

In [ ]:
model = 'PLAFPJMINORISTA'   # Nombre del modelo
partition = PERIODO         # Periodo definido en la celda de entradas

arguments = [
    '--model', model,
    '--table-score', f'scr_{model}',
    '--partition', partition,
]

processor.run(
    code=path_script,
    inputs=inputs,
    outputs=outputs,
    arguments=arguments,
)


INFO:sagemaker:Creating processing-job with name sagemaker-python3-2026-06-02-20-40-01-548


.........../usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/io/parquet/arrow.py:144: FutureWarning: 'Parque

## Configuración del Job — Modelo v2

### Variables del modelo (26 features)
| # | Variable | Importancia |
|---|----------|------------|
| 1 | `cod_ubigeo_cd` | 10.54% |
| 2 | `cnt_trx_cargostot_3m` | 9.81% |
| 3 | `mto_pas_soles` | 9.45% |
| 4 | `cnt_trx_abonospromtot_3m` | 8.77% |
| 5 | `imp_trx_abonosefect_6m` | 7.30% |
| 6 | `imp_trx_cargosefe_6m` | 5.86% |
| 7 | `rat_trx_abonosefectot_3m` | 4.92% |
| 8 | `cnt_meses_sinegresos_12m` | 4.80% |
| 9 | `mto_fact_declarado_sunat` | 4.62% |
| 10 | `num_antiguedad` | 4.61% |
| 11 | `rat_mntcrgsefetot_1m` | 3.48% |
| 12 | `avg_trx_cargostot_3m` | 3.38% |
| 13 | `cnt_alerta_hist` | 3.35% |
| 14 | `mto_del_ext_12m` | 2.84% |
| 15 | `cod_sectorista_id` | 2.80% |
| 16 | `avg_cpmenegr_12m` | 2.29% |
| 17 | `share_cp_egresos` *(derivada)* | 1.61% |
| 18 | `cnt_ros_hist` | 1.53% |
| 19 | `ratio_egresos_exterior` *(derivada)* | 1.27% |
| 20 | `mto_al_ext_12m` | 1.19% |
| 21 | `avg_cp_men_ing_12m` | 1.16% |
| 22 | `flg_alerta_12m` | 1.04% |
| 23 | `flg_vrcn_abonos_5m_1m` | 0.95% |
| 24 | `share_cp_ingresos` *(derivada)* | 0.76% |
| 25 | `cnt_noticias` | 0.66% |
| 26 | `cnt_trx_sinenv_alext_12m` | 0.52% |

### Puntos de corte
| Umbral | Valor |
|--------|-------|
| Nueva alerta (sin alerta activa) | `0.992137` |
| P1 \| P2 | `0.746` |
| P2 \| P3 | `0.909` |
| P3 \| P4 | `0.963` |
| P4 \| P5 | `0.987` |

### Artefacto
- **Modelo:** `s3://.../MODEL/calibrado_v2/model.tar.gz`
- **Datos inferencia:** `s3://.../DATA_INFERENCIA_PILOTO/INFERENCIA/periodo={PERIODO}/`
- **Instancia:** `ml.m5.12xlarge` (48 CPU, 192 GB RAM)

### Librerías disponibles en la imagen Docker
```
xgboost==1.6.2  |  pandas==1.5.1  |  numpy==1.23.5
scikit-learn==1.1.2  |  pyarrow==10.0.0  |  dask==2.11.0
```
